# biyo-cozucu · Qwen3.5 metin LoRA (Colab A100)

Bu defter `biyo-cozucu disa-aktar` ile üretilen `train.jsonl` / `valid.jsonl`
dosyalarıyla bir LoRA adaptörü eğitir, birleştirilmiş modeli dışa aktarır ve
**duman testi** yapar.

## Neden duman testi zorunlu

Qwen3.5 + LoRA birleştirme → MLX dönüşümü yolunda bilinen iki açık sorun var
(mlx-lm #1058: birleştirme sonrası ilk 20-30 token doğru, sonra sapma;
mlx-vlm #824: görsel LoRA eğitimi görüntüsüz koşuyor). Bu yüzden:

1. Eğitim **yalnız metin** — görsel katmanlara dokunulmuyor.
2. Aşağıdaki hücre, eğitim sonrası model ile diskten yeniden yüklenen
   birleştirilmiş modelin aynı prompt'ta aynı çıktıyı verip vermediğini
   karşılaştırıyor. Sapma varsa MLX'e geçmeden önce burada görülür.
3. MLX bozuksa yedek yol GGUF + Ollama (son hücre).

**Çalışma zamanı:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **A100**.

In [ ]:
# Kurulum. SÜRÜMLER BİLİNÇLİ SABİT.
# unsloth 2026.7.5 `transformers<=5.5.0`, `trl<=0.24.0` ve `datasets<4.4.0`
# istiyor. `--upgrade` ile en yenileri kurulduğunda pip yalnız bir uyumsuzluk
# uyarısı basıyor, kurulum "başarılı" görünüyor ve hata ancak saatler sonra
# `egitici.train()` çağrısında patlıyor:
#   NameError: name 'args' is not defined
# (unsloth'un ürettiği unsloth_compiled_module_qwen3_5.py içinde).
# Sürümü değiştirmen gerekirse önce unsloth'un kısıtlarına bak.
%pip install -q --upgrade unsloth unsloth_zoo
%pip install -q "transformers==5.5.0" "trl==0.24.0" "datasets==4.3.0" peft accelerate bitsandbytes

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK")
assert torch.cuda.is_available(), "GPU çalışma zamanı seçilmemiş."


## 1. Veriyi yükle

Sol paneldeki dosya simgesinden `train.jsonl` ve `valid.jsonl` dosyalarını
yükle, ya da Drive'ı bağlayıp yolu değiştir.

In [ ]:
VERI_DIZINI = "/content"   # Drive için: "/content/drive/MyDrive/biyo-cozucu/v1"
# from google.colab import drive; drive.mount("/content/drive")

import json, os
from datasets import load_dataset

for ad in ("train.jsonl", "valid.jsonl"):
    yol = os.path.join(VERI_DIZINI, ad)
    assert os.path.isfile(yol), f"{yol} bulunamadı — dosyayı yükle."

veri = load_dataset(
    "json",
    data_files={
        "train": os.path.join(VERI_DIZINI, "train.jsonl"),
        "validation": os.path.join(VERI_DIZINI, "valid.jsonl"),
    },
)
print(veri)
print("\n--- ilk kayıt ---")
for mesaj in veri["train"][0]["messages"]:
    print(f"[{mesaj['role']}] {mesaj['content'][:220]}")

## 2. Modeli yükle

Geliştirme/deneme için 4B, gerçek koşu için 9B. 4B ile bir tur tamamlayıp
hattın çalıştığını gördükten sonra 9B'ye geçmek en ucuz yol.

In [ ]:
# Veri 1.384 → 9.400 örneğe çıktı (7.525 eğitim). Hiperparametreler buna göre:
#  · AZAMI_UZUNLUK 4096 → 6144: yeni korpusta en uzun kayıt ~4.500 token ve
#    4096 sınırı 8 kaydı sessizce kırpardı. Sessiz kırpma çözümün SONUNU keser
#    ve model yarım çözüm üretmeyi öğrenir — bu yüzden pay bırakıldı.
#  · r 16 → 32: 7 kat veri daha fazla kapasiteyi taşır; r=16 bu boyutta dar.
MODEL_ADI = "unsloth/Qwen3.5-4B"   # A100 40GB'da 9B de sığar: "unsloth/Qwen3.5-9B"
AZAMI_UZUNLUK = 6144

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ADI,
    max_seq_length=AZAMI_UZUNLUK,
    load_in_4bit=True,      # 4B/9B dense; MoE modellerde bunu False yapıp bf16 kullan
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,          # alpha = 2r kuralı korundu
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=1086,
)
print(model.print_trainable_parameters())


## 3. Sohbet şablonunu uygula

Kayıtlar zaten `{"messages": [...]}` biçiminde; tokenizer'ın kendi şablonu
kullanılır. `enable_thinking=False`: eğitim verisinde düşünce bloğu yok, açık
bırakmak modele boş düşünce üretmeyi öğretir.

In [ ]:
def bicimle(kayitlar):
    metinler = []
    for mesajlar in kayitlar["messages"]:
        try:
            metin = tokenizer.apply_chat_template(
                mesajlar, tokenize=False, enable_thinking=False
            )
        except TypeError:
            metin = tokenizer.apply_chat_template(mesajlar, tokenize=False)
        metinler.append(metin)
    return {"text": metinler}

veri = veri.map(bicimle, batched=True, remove_columns=["messages"])
print(veri["train"][0]["text"][:900])

# TÜM küme ölçülüyor, ilk 200 değil: korpus artık 7.525 kayıt ve heterojen
# (on ayrı kaynaktan geldi). İlk 200 örnek uzun kuyruğu temsil etmiyor.
uzunluklar = [len(tokenizer(x)["input_ids"]) for x in veri["train"]["text"]]
uzunluklar.sort()
print(f"\nToken uzunluğu — ort {sum(uzunluklar)/len(uzunluklar):.0f}, "
      f"%95 {uzunluklar[int(len(uzunluklar)*0.95)]}, en uzun {uzunluklar[-1]}")
assert max(uzunluklar) < AZAMI_UZUNLUK, (
    "Kayıtlar AZAMI_UZUNLUK'u aşıyor: sessiz kırpma çözümün sonunu keser. "
    "AZAMI_UZUNLUK'u artır ya da dışa aktarımda --kanit-sayisi düşür."
)

## 4. Eğit

`train_on_responses_only`: kayıp yalnız asistan yanıtından hesaplanır. Soru ve
referans örnekler girdi olduğundan onları da öğrenmeye çalışmak, modeli soru
üretmeye yönlendirip asıl hedefi (çözüm üretme) zayıflatır.

In [ ]:
from trl import SFTConfig, SFTTrainer

# 7.525 eğitim kaydı için ayarlar (önceki sürüm 1.113 kayıttı):
#  · epoch 3 → 2: 7 kat veriyle 3 epoch hem gereksiz hem ezberleme riski.
#    2 epoch ≈ 940 adım (7525 / 16 × 2).
#  · efektif yığın 8 → 16 (4 × 4): A100 40GB bunu 6144 uzunlukta taşıyor;
#    daha büyük yığın gradyan gürültüsünü azaltır.
#  · eval/save adımları ölçeklendi: 25 adımda bir değerlendirme 940 adımlık
#    koşuda 37 kez durdurur ve süreyi belirgin uzatır.
#  · save_steps + load_best_model_at_end: uzun koşuda oturum düşerse ilerleme
#    kaybolmasın ve en iyi doğrulama noktası korunsun.
egitici = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=veri["train"],
    eval_dataset=veri["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=AZAMI_UZUNLUK,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        bf16=True,
        seed=1086,
        output_dir="ciktilar",
        report_to="none",
    ),
)


## 5. Eğitim sonrası hızlı bakış

Model gerçekten JSON çözüm üretiyor mu? Bu çıktıyı sakla — bir sonraki hücre
diskten yüklenen modelin çıktısını buna karşı karşılaştıracak.

In [ ]:
DENEME_SORULARI = [
    "Mayoz bölünme sonucu oluşan hücrelerde kromozom sayısı neden yarıya iner?",
    "Enzimlerin çalışmasında pH'ın optimum değerden uzaklaşması neye yol açar?",
    "Bir bitkinin ışıksız ortamda bırakılması fotosentez hızını nasıl etkiler?",
]
SISTEM = veri_sistem = None
try:
    import json as _json
    ham = open(os.path.join(VERI_DIZINI, "train.jsonl"), encoding="utf-8").readline()
    ilk = _json.loads(ham)["messages"]
    SISTEM = ilk[0]["content"] if ilk[0]["role"] == "system" else None
except Exception as hata:
    print("Sistem istemi okunamadı:", hata)

def uret(model, tokenizer, soru, azami=400):
    mesajlar = ([{"role": "system", "content": SISTEM}] if SISTEM else []) + [
        {"role": "user", "content": soru}
    ]
    try:
        istem = tokenizer.apply_chat_template(
            mesajlar, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        istem = tokenizer.apply_chat_template(
            mesajlar, tokenize=False, add_generation_prompt=True
        )
    girdiler = tokenizer(istem, return_tensors="pt").to(model.device)
    with torch.no_grad():
        cikti = model.generate(**girdiler, max_new_tokens=azami, do_sample=False)
    return tokenizer.decode(cikti[0][girdiler["input_ids"].shape[1]:],
                            skip_special_tokens=True)

FastLanguageModel.for_inference(model)
egitim_ciktilari = []
for soru in DENEME_SORULARI:
    cevap = uret(model, tokenizer, soru)
    egitim_ciktilari.append(cevap)
    print(f"\n>>> {soru}\n{cevap[:600]}")

## 6. Dışa aktar (adaptör + birleştirilmiş 16-bit)

In [ ]:
ETIKET = "biyo-lora-v1"

model.save_pretrained(f"{ETIKET}-adapter")
tokenizer.save_pretrained(f"{ETIKET}-adapter")
print("Adaptör yazıldı.")

# MLX dönüşümü birleştirilmiş 16-bit model ister.
model.save_pretrained_merged(f"{ETIKET}-merged", tokenizer, save_method="merged_16bit")
print("Birleştirilmiş model yazıldı.")
!du -sh {ETIKET}-adapter {ETIKET}-merged

## 7. DUMAN TESTİ — birleştirme çıktıyı bozdu mu?

Diskten yeniden yüklenen birleştirilmiş model, eğitim sonrası modelle aynı
çıktıyı vermeli. Vermiyorsa sorun MLX'te değil, birleştirmededir; MLX'e geçmeden
burada durdur.

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

del model, egitici
gc.collect(); torch.cuda.empty_cache()

birlesik_tokenizer = AutoTokenizer.from_pretrained(f"{ETIKET}-merged")
birlesik = AutoModelForCausalLM.from_pretrained(
    f"{ETIKET}-merged", torch_dtype=torch.bfloat16, device_map="cuda"
)

sapma = 0
for soru, egitim_cikti in zip(DENEME_SORULARI, egitim_ciktilari):
    yeni = uret(birlesik, birlesik_tokenizer, soru)
    # İlk 200 karakter aynı ise birleştirme sağlam kabul edilir; sapma tipik
    # olarak ilk 20-30 token'dan sonra başlıyor.
    ayni = yeni[:200].strip() == egitim_cikti[:200].strip()
    sapma += 0 if ayni else 1
    print(f"{'✓' if ayni else '✗'} {soru[:60]}")
    if not ayni:
        print(f"   eğitim : {egitim_cikti[:180]}")
        print(f"   birleşik: {yeni[:180]}")

print(f"\nSapan soru: {sapma}/{len(DENEME_SORULARI)}")
if sapma:
    print("UYARI: birleştirme çıktıyı değiştiriyor. MLX'e geçme; "
          "adaptörü ayrı tutup mlx_lm.fuse dene ya da GGUF yoluna geç.")
else:
    print("Birleştirme sağlam. MLX dönüşümüne geçilebilir.")

## 8. Paketle ve indir

In [ ]:
!zip -qr {ETIKET}-merged.zip {ETIKET}-merged
!zip -qr {ETIKET}-adapter.zip {ETIKET}-adapter
!ls -lh {ETIKET}-*.zip

from google.colab import files
files.download(f"{ETIKET}-adapter.zip")   # küçük; önce bunu indir
# files.download(f"{ETIKET}-merged.zip")  # birkaç GB — Drive'a kopyalamak daha hızlı
# !cp {ETIKET}-merged.zip /content/drive/MyDrive/

## 9. Mac'te devam

**Adaptörü indir, birleştirmeyi Mac'te yap.** Birleştirilmiş model 8,7 GB;
adaptör 79 MB. Tarayıcıdan 8,7 GB indirmek kırılgan ve gereksiz.

```bash
biyo-cozucu model-kur ~/Downloads/biyo-lora-v1-adapter.zip

# Adaptörü taban modelle birleştir (taban model HF'den çekilir, ~8 GB)
uv run python scripts/lora_birlestir.py

# MLX'e çevir (4-bit, ~2,2 GB)
uv run python -m mlx_lm.convert \
  --hf-path ~/.biyo-cozucu/modeller/biyo-lora-v1-fused \
  --mlx-path ~/.biyo-cozucu/modeller/biyo-mlx-v1 -q --q-bits 4 --q-group-size 64

export BIYO_MLX_MODEL=~/.biyo-cozucu/modeller/biyo-mlx-v1
biyo-cozucu degerlendir --ayrintili        # kör koşu: taban modelle karşılaştır
```

`scripts/lora_birlestir.py` model sınıfını `Qwen3_5ForConditionalGeneration`
olarak sabitliyor ve birleşmeyi bir tanık ağırlıkla doğruluyor: adaptör
anahtarları `...model.language_model.layers...` yolunda olduğu için
`AutoModelForCausalLM` hiçbirini eşleştiremiyor, PEFT yalnız `UserWarning`
verip taban modelin kopyasını yazıyor.

`degerlendir` varsayılan olarak sıcaklık 0 ile koşar; koşular arası fark ancak
böyle ölçülebilir. Önbellek anahtarı model kimliğini içerir, yani eğitilmiş
model taban modelin kayıtlı cevaplarını geri döndürmez.

### MLX bozuksa yedek yol (GGUF + Ollama)

```python
model.save_pretrained_gguf(ETIKET, tokenizer, quantization_method="q4_k_m")
```

Mac'te `ollama create biyo -f Modelfile && ollama serve`, sonra
`BIYO_MOTOR=http BIYO_MOTOR_URL=http://127.0.0.1:11434/v1 BIYO_MOTOR_MODELLER=biyo`.
